# 04 - Standardization

## Objective

Membuat representasi terstandardisasi untuk eksperimen deterministic matching tanpa mengubah raw dataset.

## Research Questions

1. Apakah standardisasi mengurangi variasi formatting pada field identitas?
2. Apakah standardisasi meningkatkan collision yang dapat dipakai sebagai blocking key?
3. Apakah ada nilai yang menjadi invalid atau hilang setelah standardisasi?

## Hypothesis

- Casefold dan trimming akan menyatukan variasi email yang hanya berbeda format.
- Pengambilan digit akan membuat telepon dapat dibandingkan secara konsisten, tetapi tidak otomatis membuktikan nomor valid.
- Normalisasi nama dan alamat dapat meningkatkan recall candidate generation, sehingga collision harus diaudit sebelum dipakai untuk matching.

## Decision boundary

Standardisasi hanya membuat kolom turunan. Tidak ada baris yang dihapus, tidak ada raw column yang ditimpa, dan tidak ada fuzzy matching pada tahap ini.

In [1]:
from pathlib import Path
import pandas as pd

DATA_CANDIDATES = [
    Path.cwd() / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
    Path.cwd().parent / 'data' / 'raw' / 'crm_50000_customers_dirty_v3.csv',
]
DATA_PATH = next((path.resolve() for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('CSV dataset tidak ditemukan.')

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()
raw_columns = raw_df.columns.tolist()
print(f'File: {DATA_PATH}')
print(f'Shape: {df.shape}')

File: C:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\raw\crm_50000_customers_dirty_v3.csv
Shape: (50000, 14)


## Standardization rules

- Text identity fields: Unicode normalization `NFKC`, casefold, whitespace collapse, dan trim.
- Email: aturan text identity ditambah penghapusan whitespace di sekitar nilai; tidak mengubah domain atau local-part secara semantik.
- Phone: simpan digit saja; tidak menambahkan country code karena aturan negara belum tersedia.
- Name key: gabungan nama terstandardisasi dengan karakter non-alphanumeric dihapus untuk blocking analisis.
- Address: hanya standardisasi whitespace dan case; singkatan alamat belum diubah.
- Date: parse dengan `pandas.to_datetime` lalu simpan sebagai ISO date `YYYY-MM-DD`.

Semua aturan bersifat deterministic dan dapat diulang.

In [2]:
def standardize_text(series: pd.Series) -> pd.Series:
    return (
        series.astype('string')
        .str.normalize('NFKC')
        .str.casefold()
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
        .replace('', pd.NA)
    )

df['email_std'] = standardize_text(df['email'])
df['phone_digits_std'] = (
    df['phone_number'].astype('string')
    .str.normalize('NFKC')
    .str.replace(r'\D', '', regex=True)
    .replace('', pd.NA)
)
df['first_name_std'] = standardize_text(df['first_name'])
df['last_name_std'] = standardize_text(df['last_name'])
df['name_key_std'] = (
    df['first_name_std'].fillna('') + df['last_name_std'].fillna('')
).str.replace(r'[^a-z0-9]', '', regex=True).replace('', pd.NA)
df['address_std'] = standardize_text(df['address'])
df['dob_std'] = pd.to_datetime(df['dob'], errors='coerce').dt.strftime('%Y-%m-%d')
df['signup_date_std'] = pd.to_datetime(df['signup_date'], errors='coerce').dt.strftime('%Y-%m-%d')

standardized_columns = [
    'email_std', 'phone_digits_std', 'first_name_std', 'last_name_std',
    'name_key_std', 'address_std', 'dob_std', 'signup_date_std',
]
print(f'Added standardized columns: {len(standardized_columns)}')
print('Standardized columns:', standardized_columns)
print('Standardized missing counts:')
print(df[standardized_columns].isna().sum().to_string())

Added standardized columns: 8
Standardized columns: ['email_std', 'phone_digits_std', 'first_name_std', 'last_name_std', 'name_key_std', 'address_std', 'dob_std', 'signup_date_std']
Standardized missing counts:
email_std           1040
phone_digits_std       0
first_name_std         0
last_name_std          0
name_key_std           0
address_std            0
dob_std                0
signup_date_std        0


In [3]:
# Validasi bahwa raw columns tetap sama dan jumlah baris tidak berubah
raw_columns_unchanged = df[raw_columns].equals(raw_df[raw_columns])
row_count_unchanged = len(df) == len(raw_df)

validation_summary = pd.DataFrame({
    'check': ['raw_columns_unchanged', 'row_count_unchanged', 'raw_column_count', 'standardized_column_count'],
    'result': [raw_columns_unchanged, row_count_unchanged, len(raw_columns), len(standardized_columns)],
})
validation_summary

,check,result
0,raw_columns_unchanged,True
1,row_count_unchanged,True
2,raw_column_count,14
3,standardized_column_count,8


## Experiment - Before/after uniqueness and missingness

Perbandingan ini mengukur dampak standardisasi, bukan kualitas entity resolution. Penurunan uniqueness menunjukkan collision bertambah dan harus diaudit pada tahap matching.

In [4]:
comparison_pairs = [
    ('email', 'email_std'),
    ('phone_number', 'phone_digits_std'),
    ('address', 'address_std'),
    ('dob', 'dob_std'),
    ('signup_date', 'signup_date_std'),
]

comparison_rows = []
for raw_column, standardized_column in comparison_pairs:
    comparison_rows.append({
        'raw_column': raw_column,
        'standardized_column': standardized_column,
        'raw_missing': int(raw_df[raw_column].isna().sum()),
        'standardized_missing': int(df[standardized_column].isna().sum()),
        'raw_unique': int(raw_df[raw_column].nunique(dropna=True)),
        'standardized_unique': int(df[standardized_column].nunique(dropna=True)),
        'unique_change': int(df[standardized_column].nunique(dropna=True) - raw_df[raw_column].nunique(dropna=True)),
    })

standardization_comparison = pd.DataFrame(comparison_rows)
standardization_comparison

,raw_column,standardized_column,raw_missing,standardized_missing,raw_unique,standardized_unique,unique_change
0,email,email_std,1040,1040,46363,46363,0
1,phone_number,phone_digits_std,0,0,46777,46777,0
2,address,address_std,0,0,48200,48200,0
3,dob,dob_std,0,0,17733,17733,0
4,signup_date,signup_date_std,0,0,4383,4383,0


In [5]:
# Audit collision size pada blocking key yang akan dipakai berikutnya
name_dob_parts = df[['name_key_std', 'dob_std']]
df['name_dob_key_std'] = name_dob_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[name_dob_parts.isna().any(axis=1), 'name_dob_key_std'] = pd.NA

email_phone_parts = df[['email_std', 'phone_digits_std']]
df['email_phone_key_std'] = email_phone_parts.fillna('').astype('string').agg('|'.join, axis=1)
df.loc[email_phone_parts.isna().any(axis=1), 'email_phone_key_std'] = pd.NA

blocking_key_columns = ['email_std', 'phone_digits_std', 'name_dob_key_std', 'email_phone_key_std']
blocking_rows = []
for column in blocking_key_columns:
    counts = df[column].value_counts(dropna=True)
    repeated = counts[counts > 1]
    blocking_rows.append({
        'blocking_key': column,
        'unique_keys': int(df[column].nunique(dropna=True)),
        'repeated_blocks': int(len(repeated)),
        'rows_in_repeated_blocks': int(repeated.sum()),
        'candidate_pairs': int((repeated * (repeated - 1) // 2).sum()),
        'largest_block': int(repeated.max()) if len(repeated) else 0,
    })

blocking_summary = pd.DataFrame(blocking_rows).sort_values('candidate_pairs')
blocking_summary

,blocking_key,unique_keys,repeated_blocks,rows_in_repeated_blocks,candidate_pairs,largest_block
2,name_dob_key_std,48565,1388,2823,1482,3
3,email_phone_key_std,47200,1695,3455,1826,4
0,email_std,46363,2102,4699,3422,10
1,phone_digits_std,46777,2161,5384,5509,9


In [6]:
# Simpan hasil terstandardisasi secara terpisah dari raw dataset
OUTPUT_DIR = DATA_PATH.parents[1] / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / 'crm_50000_customers_standardized.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved transformed dataset: {OUTPUT_PATH}')
print(f'Output shape: {df.shape}')
print(f'Raw dataset still exists: {DATA_PATH.exists()}')

Saved transformed dataset: C:\Users\User\Documents\Maganghub 2026\Bulan 1\Tes duplikasi\data\processed\crm_50000_customers_standardized.csv
Output shape: (50000, 24)
Raw dataset still exists: True


# Result, Analysis, and Decision

## Interpretation

- `*_std` adalah representasi analisis, bukan pengganti kolom raw.
- Collision yang meningkat setelah standardisasi tidak otomatis berarti duplicate entity.
- `phone_digits_std`, `email_std`, `name_dob_key_std`, dan `email_phone_key_std` menjadi kandidat blocking untuk tahap deterministic matching.
- Threshold fuzzy belum diperlukan karena standardisasi dan deterministic matching belum diuji.

## Limitations

- Country code telepon tidak ditambahkan karena aturan negara belum ditetapkan.
- Alamat belum distandardisasi menggunakan kamus singkatan.
- Tidak ada ground truth untuk menilai precision dan recall.
- Tanggal berhasil diparse, tetapi ambiguitas format tanggal tidak dapat dibuktikan hanya dari hasil parse.

## Decision

Gunakan kolom terstandardisasi untuk eksperimen deterministic matching. Jangan menghapus duplicate dan jangan menimpa raw dataset.

## Next Experiment

Tahap berikutnya adalah `05_deterministic_matching.ipynb`, dengan aturan matching yang eksplisit dan audit candidate pair. Fuzzy matching belum dijalankan sebelum hasil deterministic matching dianalisis.